# Step 3 · Model 2: Logistic regression (readable)

**Goal:** give every variable a weight, so a campaign's score is built by adding and subtracting points.
A higher goal loses points, a long campaign loses points, a strong category gains points. The total turns
into a 0–100% chance of success — and you can read exactly how much each variable pushed it.

**Inputs:** all 191 encoded columns from Step 2. **Scaler fitted on training (2019–23) only** (L4 slide 17:
using validation or test statistics leaks the future into the inputs).

**Results on validation (2024), filled in after running this notebook:**

| | |
|---|---|
| **AUC** | **0.853** — beats Model 1's 0.807 by +0.046 ✅ |
| Within "Games / Tabletop Games" alone | AUC 0.776 (Model 1: 0.500 by construction) |
| Best pre-declared coaching bracket (40–60%) | 4.21% near-miss hit rate, **2.08× random**, 2,542 campaigns |
| Near-miss hit rate, riskiest-first | 0.95% vs 2.02% random — **0.47×, worse than Model 1** |
| Calibration | +6.4 points optimistic |

**Two headlines:**

1. **Model 2 beats the baseline**, and section 6 shows *where* the gain comes from: it can rank campaigns
   *inside* a category, which a lookup table cannot do at all.
2. **Model 2 is better at AUC and worse at the naive coaching list.** Riskiest-first drops from 1.85%
   (Model 1) to 0.95%. This is not a bug — it is the Model 1 finding getting sharper, and section 9
   explains why a *better* model makes that particular rule *worse*.

**Team rules followed:** launch-day inputs only (no `post_` columns), scaler and model fitted on training
only, 2025 test file never opened.

### PIPELINE — Load the encoded inputs and the answers
`X_train` / `X_val` hold the 191 model inputs; the answers (`success`, `near_miss`) live in the `slice_`
files. Step 2 built them as matching row orders, and the next cell **checks that** rather than trusting it —
if the rows were misaligned every campaign would be scored against someone else's answer and nothing
downstream would warn us.

Checks: 191 identical columns, 0 blanks, and three input columns that appear in both files match row for row.

In [1]:
import os
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix

BASE = "/content/split"
if not os.path.isdir(BASE):
    BASE = "."

X_train = pd.read_parquet(f"{BASE}/X_train.parquet")
X_val   = pd.read_parquet(f"{BASE}/X_val.parquet")
train   = pd.read_parquet(f"{BASE}/slice_train.parquet")
val     = pd.read_parquet(f"{BASE}/slice_val.parquet")

y_train = train["success"].to_numpy()
y_val   = val["success"].to_numpy()
nm_val  = val["near_miss"].to_numpy()

assert list(X_train.columns) == list(X_val.columns), "column mismatch"
for nm_, X, S in [("train", X_train, train), ("val", X_val, val)]:
    assert len(X) == len(S), f"{nm_}: row count mismatch"
    for c in ["log_goal_usd", "duration_days", "name_len"]:
        assert np.allclose(X[c].to_numpy(float), S[c].to_numpy(float)), f"{nm_}: rows misaligned on {c}"

print(f"X_train {X_train.shape}   X_val {X_val.shape}")
print(f"identical columns: {list(X_train.columns) == list(X_val.columns)}   "
      f"blanks: {int(X_train.isna().sum().sum() + X_val.isna().sum().sum())}")
print("row alignment with the slice_ files: verified on 3 shared columns")
print(f"\ntraining success rate {y_train.mean():.1%}   validation success rate {y_val.mean():.1%}")

X_train (119297, 191)   X_val (26739, 191)
identical columns: True   blanks: 0
row alignment with the slice_ files: verified on 3 shared columns

training success rate 73.5%   validation success rate 72.2%


### PIPELINE — Load Model 1's scores for the comparison
Model 1 saved `baseline_val.parquet` precisely so every later model compares on identical rows rather than
rebuilding the baseline and hoping it matches. If the file is missing the cell rebuilds the lookup, which is
one line.

In [2]:
try:
    m1 = pd.read_parquet(f"{BASE}/baseline_val.parquet")
    assert (m1["id"].to_numpy() == val["id"].to_numpy()).all(), "baseline rows do not match slice_val"
    score_m1 = m1["score_model1"].to_numpy()
    print("loaded baseline_val.parquet from Model 1")
except FileNotFoundError:
    lookup = train.groupby("subcategory")["success"].mean()
    score_m1 = val["subcategory"].map(lookup).fillna(y_train.mean()).to_numpy()
    print("baseline_val.parquet not found - rebuilt the Model 1 lookup inline")

AUC_MODEL1 = roc_auc_score(y_val, score_m1)
print(f"Model 1 AUC on 2024 = {AUC_MODEL1:.4f}   <- the bar")

loaded baseline_val.parquet from Model 1
Model 1 AUC on 2024 = 0.8067   <- the bar


### PIPELINE — Standardize, fitting the scaler on training only
Logistic regression gives each column one weight per unit, so columns on wildly different scales get
incomparable weights: `log_goal_usd` runs 0.4–18.6 while `blurb_len` runs 0–370. Standardizing puts every
column on the same footing — zero mean, one standard deviation — so the weights can be compared directly.

**`.fit(X_train)` then `.transform(...)` on each slice.** The mean and standard deviation come from training
only. Fitting the scaler on validation would quietly feed 2024 into the model's inputs (L4 slide 17), the
same leak Step 2 avoided when it learned the top-60 category list from training alone.

In [3]:
scaler = StandardScaler().fit(X_train)       # training only
X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_val)

print(f"training means after scaling   : {X_train_s.mean():.2e}  (≈ 0 by construction)")
print(f"training std devs after scaling: {X_train_s.std():.4f}  (≈ 1 by construction)")
print(f"validation means after scaling : {X_val_s.mean():+.4f}  (NOT 0 - and that is correct:")
print( "                                  2024 differs from 2019-23, and we want to see that)")

training means after scaling   : 2.54e-18  (≈ 0 by construction)


training std devs after scaling: 1.0000  (≈ 1 by construction)
validation means after scaling : -0.0003  (NOT 0 - and that is correct:
                                  2024 differs from 2019-23, and we want to see that)


### EXPLORATION — How hard should the model be regularized?
`C` controls how much the model is allowed to trust any single column: small `C` = strong regularization =
weights pulled toward zero. Validation is the right place to choose it (L4 slide 24: training calibrates,
validation selects, test stays in the vault).

**It barely matters.** Validation AUC moves from 0.8509 to 0.8526 across four orders of magnitude — a
0.0017 spread that is well inside noise. We take `C = 0.001` because validation says so, but the honest
reading is that **this model is insensitive to regularization**, and HW4's default `C = 1.0` would give
0.8509.

Worth noting the **train 0.922 vs validation 0.853 gap**. Some is genuine overfitting; some is year-to-year
drift, since 2024 is simply a different year from 2019–23. Model 1 had the same kind of gap and no fitted
parameters at all, which is the clue that drift is a real part of it.

In [4]:
print(f"{'C':>8}{'train AUC':>12}{'val AUC':>10}{'largest |weight|':>18}")
print("-" * 48)
sweep = {}
for C in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    m = LogisticRegression(max_iter=3000, C=C).fit(X_train_s, y_train)
    a_tr = roc_auc_score(y_train, m.predict_proba(X_train_s)[:, 1])
    a_va = roc_auc_score(y_val,   m.predict_proba(X_val_s)[:, 1])
    sweep[C] = a_va
    print(f"{C:>8}{a_tr:>12.4f}{a_va:>10.4f}{np.abs(m.coef_[0]).max():>18.2f}")

C_BEST = max(sweep, key=sweep.get)
print(f"\nbest on validation: C = {C_BEST}  (spread across the whole sweep: "
      f"{max(sweep.values()) - min(sweep.values()):.4f})")

       C   train AUC   val AUC  largest |weight|
------------------------------------------------


   0.001      0.9220    0.8526              0.69


    0.01      0.9228    0.8514              0.79


     0.1      0.9229    0.8510              0.82


     1.0      0.9229    0.8509              0.84


    10.0      0.9229    0.8509              0.84


   100.0      0.9229    0.8509              0.84

best on validation: C = 0.001  (spread across the whole sweep: 0.0017)


### PIPELINE — Fit Model 2
`max_iter=3000` follows HW4 Part 4. The model converges in 18 iterations, so the limit is never binding.

In [5]:
model = LogisticRegression(max_iter=3000, C=C_BEST).fit(X_train_s, y_train)
score_m2 = model.predict_proba(X_val_s)[:, 1]

AUC_MODEL2 = roc_auc_score(y_val, score_m2)
print(f"converged in {model.n_iter_[0]} iterations (limit 3000)\n")
print(f"Model 1 (category lookup)      AUC = {AUC_MODEL1:.4f}")
print(f"Model 2 (logistic, 191 inputs) AUC = {AUC_MODEL2:.4f}")
print(f"gain over the baseline             = {AUC_MODEL2 - AUC_MODEL1:+.4f}   "
      f"{'BEATS the baseline' if AUC_MODEL2 > AUC_MODEL1 else 'DOES NOT beat the baseline'}")

converged in 18 iterations (limit 3000)

Model 1 (category lookup)      AUC = 0.8067
Model 2 (logistic, 191 inputs) AUC = 0.8526
gain over the baseline             = +0.0459   BEATS the baseline


### PIPELINE — Turn the weights back into plain English
The model is fitted on standardized inputs, so its raw weights read as "per 1 standard deviation", which is
fine for comparing columns but useless for talking to a creator. Dividing each weight by that column's
training standard deviation puts it back into **original units** — per dollar of goal, per day of campaign.

A logistic regression adds up points on the **log-odds** scale, so a weight of `b` on a variable means:
change that variable by one unit and the *odds* of success multiply by `exp(b)`. That multiplier is what
the next two cells report.

⚠️ **One caveat on reading individual category weights.** Step 2 kept *all* levels of each categorical
variable (15 parent categories, not 14), where L2 slide 28 creates `m − 1` dummies and leaves one out as a
reference. With all levels plus an intercept the columns are collinear; the L2 penalty still gives a unique
answer, but each category's weight reads as **"versus the average category"** rather than "versus a named
reference level". That is arguably easier to explain, but it is not the textbook setup, and anyone reading
the table needs to know which one they are looking at.

In [6]:
coef_std = model.coef_[0]                       # per 1 standard deviation
coef_raw = pd.Series(coef_std / scaler.scale_, index=X_train.columns)   # per 1 original unit
# intercept, moved back to the original scale
b0 = model.intercept_[0] - float((coef_std / scaler.scale_ * scaler.mean_).sum())

NUMERIC = ["log_goal_usd", "duration_days", "log_prep_days", "name_len", "blurb_len"]
plain = pd.DataFrame({
    "variable": ["goal: doubling it", "goal: 10x bigger", "campaign length: +10 days",
                 "campaign length: +30 days", "prep time: doubling it",
                 "name: +10 characters", "blurb: +10 characters"],
    "odds multiplier": [np.exp(coef_raw["log_goal_usd"] * np.log(2)),
                        np.exp(coef_raw["log_goal_usd"] * np.log(10)),
                        np.exp(coef_raw["duration_days"] * 10),
                        np.exp(coef_raw["duration_days"] * 30),
                        np.exp(coef_raw["log_prep_days"] * np.log(2)),
                        np.exp(coef_raw["name_len"] * 10),
                        np.exp(coef_raw["blurb_len"] * 10)]})
plain["change in odds"] = (plain["odds multiplier"] - 1).map(lambda v: f"{v:+.0%}")
print("WHAT THE LAUNCH-DAY NUMBERS DO TO THE ODDS OF SUCCESS\n")
print(plain.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"\nstarting point (intercept, original scale): {b0:+.3f} points "
      f"= {1/(1+np.exp(-b0)):.1%} before any variable is counted")

WHAT THE LAUNCH-DAY NUMBERS DO TO THE ODDS OF SUCCESS

                 variable  odds multiplier change in odds
        goal: doubling it            0.758           -24%
         goal: 10x bigger            0.398           -60%
campaign length: +10 days            0.757           -24%
campaign length: +30 days            0.434           -57%
   prep time: doubling it            1.191           +19%
     name: +10 characters            1.129           +13%
    blurb: +10 characters            1.006            +1%

starting point (intercept, original scale): +4.835 points = 99.2% before any variable is counted


### EXPLORATION — Which variables move the prediction most?
Ranked by the standardized weight, which answers "if this column moved by one standard deviation, how much
would the score move?" — the fair way to compare a dollar goal against a yes/no category flag.

The story is readable and it matches what Step 2 found by hand:
- **The funding goal is the single strongest lever** and the only numeric variable near the top. Doubling it
  cuts the odds of success by about a quarter.
- **Category still dominates the rest of the table**, which is exactly why Model 1 reached 0.807.
- **Longer campaigns are worse, more preparation is better** — Step 2 saw both patterns in raw success
  rates, and the model confirms them while holding everything else fixed.
- **`subcategory_other` is strongly negative**, which is a warning rather than a finding: "other" is 18.9%
  of training lumped into one cell, so the model is really saying "unusual categories do badly on average".

In [7]:
co = pd.DataFrame({"feature": X_train.columns, "weight_std": coef_std,
                   "per_unit": coef_raw.to_numpy(),
                   "odds_multiplier": np.exp(coef_raw.to_numpy())})

print("STRONGEST 12 PUSHING TOWARD SUCCESS\n")
print(co.nlargest(12, "weight_std").to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\n\nSTRONGEST 12 PUSHING TOWARD FAILURE\n")
print(co.nsmallest(12, "weight_std").to_string(index=False, float_format=lambda x: f"{x:.3f}"))

STRONGEST 12 PUSHING TOWARD SUCCESS

                                  feature  weight_std  per_unit  odds_multiplier
       subcategory_Games / Tabletop Games       0.453     1.552            4.719
      subcategory_Design / Product Design       0.440     1.726            5.617
                            log_prep_days       0.376     0.252            1.287
                    subcategory_Art / Art       0.342     1.901            6.691
           subcategory_Art / Illustration       0.304     1.612            5.012
         subcategory_Comics / Comic Books       0.300     1.374            3.951
subcategory_Publishing / Children's Books       0.300     2.082            8.017
                subcategory_Music / Music       0.226     2.020            7.537
        subcategory_Film & Video / Shorts       0.225     1.945            6.995
         subcategory_Publishing / Fiction       0.206     1.448            4.253
                                 name_len       0.188     0.012         

### PIPELINE — Reading one real campaign's score, line by line
This is the deliverable the plan asked for: *"you can read how much each variable pushes the prediction up
or down."* The cell takes a real 2024 campaign — one that actually ended as a near-miss — and shows the
running total, with the chance of success after each line.

Every row is one variable's contribution in points. The running total is on the log-odds scale; the last
column converts it to a probability at that point, so you can watch the score move.

Reading the output below: the campaign starts at 99.2% (the intercept — before anything is known, campaigns
mostly succeed), loses 3.7 points for a $10,658 goal, loses another 0.8 for running 30 days, gains 0.8 back
for 22 days of preparation, and then loses 1.3 for sitting in the `other` category. It lands at **45.6%** —
squarely in the coaching bracket — and it did in fact end as a near-miss.

In [8]:
window = (score_m2 > 0.40) & (score_m2 <= 0.60)
i = int(np.where(window & (nm_val == 1))[0][0])    # first near-miss inside the coaching window
row, xr = val.iloc[i], X_val.iloc[i]

print(f"campaign {row['id']} | {row['subcategory']} | goal ${row['goal_usd']:,.0f} | "
      f"{row['duration_days']:.0f} days | {row['prep_days']:.0f} days of prep")
print(f"what actually happened: success={row['success']}, near_miss={row['near_miss']}\n")

contrib = xr * coef_raw
parts = [("starting point (intercept)", b0)]
parts += [(f"{f} = {xr[f]:,.2f}", contrib[f]) for f in NUMERIC]
parts += [(c, contrib[c]) for c in X_train.columns
          if c not in NUMERIC and xr[c] == 1]

running = 0.0
print(f"{'what the model sees':<46}{'points':>9}{'total':>9}{'chance':>9}")
print("-" * 73)
for label, v in parts:
    running += v
    print(f"{label:<46}{v:>+9.3f}{running:>9.3f}{1/(1+np.exp(-running)):>9.1%}")
print(f"\nModel 2's score for this campaign: {score_m2[i]:.1%}")

campaign 612560855 | other | goal $10,658 | 30 days | 22 days of prep
what actually happened: success=0, near_miss=1

what the model sees                              points    total   chance
-------------------------------------------------------------------------
starting point (intercept)                       +4.835    4.835    99.2%
log_goal_usd = 9.27                              -3.713    1.122    75.4%
duration_days = 30.42                            -0.848    0.274    56.8%
log_prep_days = 3.13                             +0.791    1.066    74.4%
name_len = 26.00                                 +0.315    1.381    79.9%
blurb_len = 98.00                                +0.060    1.441    80.9%
parent_category_Film & Video                     -0.032    1.409    80.4%
subcategory_other                                -1.328    0.081    52.0%
loc_state_CA / QC                                -0.075    0.005    50.1%
loc_type_Town                                    -0.017   -0.012    

### EXPLORATION — Where the gain over Model 1 actually comes from
Model 1's limitation was specific: it gives every "Games / Tabletop Games" campaign the same 99.5%, whether
the goal is $500 or $500,000. Its AUC *inside* any single category is therefore exactly 0.500 — a coin flip —
by construction.

Model 2 scores 26,739 distinct values instead of 60, and the table below shows it ranking well inside
categories where Model 1 cannot rank at all. **That is the entire +0.046.** It is worth stating this way in
the deck: the improvement is not "logistic regression is better", it is "the baseline could not see goal
size, length or prep time, and those matter".

In [9]:
print(f"distinct scores on 2024:  Model 1 = {len(np.unique(score_m1)):,}   "
      f"Model 2 = {len(np.unique(score_m2)):,}\n")

rows = []
for cat in ["Games / Tabletop Games", "Technology / Apps", "Film & Video / Documentary",
            "Publishing / Children's Books", "other"]:
    m = (val["subcategory"] == cat).to_numpy()
    if m.sum() < 50 or len(np.unique(y_val[m])) < 2:
        continue
    rows.append({"subcategory": cat, "campaigns in 2024": int(m.sum()),
                 "success rate": y_val[m].mean(),
                 "Model 1 AUC here": 0.5,
                 "Model 2 AUC here": roc_auc_score(y_val[m], score_m2[m])})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\nModel 1's within-category AUC is 0.500 by construction: one score per category cannot rank.")

distinct scores on 2024:  Model 1 = 60   Model 2 = 26,739

                  subcategory  campaigns in 2024  success rate  Model 1 AUC here  Model 2 AUC here
       Games / Tabletop Games               4907         0.954             0.500             0.776
            Technology / Apps                606         0.092             0.500             0.818
   Film & Video / Documentary                328         0.534             0.500             0.721
Publishing / Children's Books                422         0.808             0.500             0.704
                        other               3937         0.481             0.500             0.747

Model 1's within-category AUC is 0.500 by construction: one score per category cannot rank.


### PIPELINE — The coaching list, same function as Model 1
Capacity-based, not a 0.50 cutoff, for the reasons set out in Model 1: nobody decided Kickstarter can coach
however many campaigns happen to fall below 0.50. `COACH_CAPACITY` is still a placeholder awaiting a real
staffing number.

Model 2 has continuous scores, so unlike Model 1 the capacity cut lands cleanly — no ties to break
arbitrarily.

In [10]:
COACH_CAPACITY = 2_000
RANDOM_NM_RATE = nm_val.mean()
ids = val["id"].to_numpy()

def coaching_list(score, label, capacity=COACH_CAPACITY, quiet=False):
    """Fill the coaching capacity riskiest-first; report what that list caught in 2024."""
    score = np.asarray(score, dtype=float)
    called = np.lexsort((ids, score))[:capacity]
    near_misses = int(nm_val[called].sum())
    failures    = int((1 - y_val[called]).sum())
    summary = {"model": label, "near-misses caught": near_misses,
               "near-miss hit rate": near_misses / capacity,
               "lift vs random": (near_misses / capacity) / RANDOM_NM_RATE,
               "failures caught": failures, "precision": failures / capacity}
    if not quiet:
        print(f"{label}")
        print(f"  campaigns called   : {capacity:,}")
        print(f"  failures caught    : {failures:,} ({failures / capacity:.1%} of the list was right)")
        print(f"  near-misses caught : {near_misses:,}")
        print(f"  near-miss hit rate : {near_misses / capacity:.2%}  (random {RANDOM_NM_RATE:.2%}, "
              f"lift {(near_misses / capacity) / RANDOM_NM_RATE:.2f}x)")
    return summary

results_m1 = coaching_list(score_m1, "Model 1: category lookup")
print()
results_m2 = coaching_list(score_m2, "Model 2: logistic regression")

Model 1: category lookup
  campaigns called   : 2,000
  failures caught    : 1,632 (81.6% of the list was right)
  near-misses caught : 37
  near-miss hit rate : 1.85%  (random 2.02%, lift 0.91x)

Model 2: logistic regression
  campaigns called   : 2,000
  failures caught    : 1,851 (92.5% of the list was right)
  near-misses caught : 19
  near-miss hit rate : 0.95%  (random 2.02%, lift 0.47x)


### EXPLORATION — ⚠️ The better model builds a worse list (riskiest-first)
Model 2 is **better** at finding failures: 92.5% of its 2,000 calls did fail, against Model 1's 81.6%.
And it is **worse** at finding near-misses: 0.95% against 1.85%, with random at 2.02%.

Those two facts are the same fact. Model 2 can see goal size, so it identifies campaigns that are *hopeless* —
a $90,000 goal, 60 days, no preparation, in a weak category. Those campaigns do fail, at 2% of goal. They
are not near-misses and a coaching call will not save them. Model 1 was too blunt to find them, and
accidentally scraped up more borderline campaigns along the way.

**The lesson, stated for the deck: improving AUC made this particular business rule worse.** That is why
the Step 4 comparison table needs both columns, and why Step 5's rule cannot be "call the riskiest".

In [11]:
print(f"{'':<32}{'precision':>11}{'near-miss hit':>15}{'lift':>8}")
print("-" * 66)
for r in [results_m1, results_m2]:
    print(f"{r['model']:<32}{r['precision']:>11.1%}{r['near-miss hit rate']:>15.2%}"
          f"{r['lift vs random']:>7.2f}x")
print(f"{'random (call 2,000 at random)':<32}{1-y_val.mean():>11.1%}"
      f"{RANDOM_NM_RATE:>15.2%}{1.0:>7.2f}x")
print("\nbetter at spotting failures -> worse at spotting near-misses. Same fact, stated twice.")

                                  precision  near-miss hit    lift
------------------------------------------------------------------
Model 1: category lookup              81.6%          1.85%   0.91x
Model 2: logistic regression          92.5%          0.95%   0.47x
random (call 2,000 at random)         27.8%          2.02%   1.00x

better at spotting failures -> worse at spotting near-misses. Same fact, stated twice.


### EXPLORATION — With the plan's bracket rule, Model 2 wins properly
These are the five brackets written into Step 5 of the plan — declared in advance, not searched for, so the
numbers are not the product of trying many windows and reporting the best.

**Model 2's 40–60% bracket: 2,542 campaigns at a 4.21% near-miss hit rate, 2.08× random.** Model 1's best
was 3.76% on 931 campaigns. Two separate wins:
- a **higher hit rate** (4.21% vs 3.76%), and
- a bracket that is **roughly the right size** — 2,542 against a 2,000-campaign capacity. Model 1's best
  bracket held 931 campaigns and there was no way to make it bigger, because a 60-value lookup table cannot
  produce a list of any chosen size.

The near-miss rate still falls off at both ends, exactly as it did for Model 1. A sharper model moved the
peak but did not move *where* the peak is.

In [12]:
def brackets(score, label):
    d = pd.DataFrame({"score": score, "success": y_val, "near_miss": nm_val})
    d["bracket"] = pd.cut(d["score"], [0, .2, .4, .6, .8, 1.0], include_lowest=True)
    g = d.groupby("bracket", observed=False).agg(campaigns=("success", "size"),
                                                 success_rate=("success", "mean"),
                                                 near_misses=("near_miss", "sum"),
                                                 hit_rate=("near_miss", "mean"))
    g["lift"] = g["hit_rate"] / RANDOM_NM_RATE
    print(f"{label}   (random hit rate {RANDOM_NM_RATE:.2%})")
    print(g.to_string(float_format=lambda x: f"{x:.4f}"), "\n")
    return g

b1 = brackets(score_m1, "MODEL 1")
b2 = brackets(score_m2, "MODEL 2")
print(f"Model 1 best bracket: {b1['hit_rate'].max():.2%} on {b1.loc[b1['hit_rate'].idxmax(),'campaigns']:,.0f} campaigns")
print(f"Model 2 best bracket: {b2['hit_rate'].max():.2%} on {b2.loc[b2['hit_rate'].idxmax(),'campaigns']:,.0f} campaigns")

MODEL 1   (random hit rate 2.02%)
               campaigns  success_rate  near_misses  hit_rate   lift
bracket                                                             
(-0.001, 0.2]       1226        0.1060           15    0.0122 0.6047
(0.2, 0.4]           931        0.3233           35    0.0376 1.8581
(0.4, 0.6]          7101        0.5164          240    0.0338 1.6705
(0.6, 0.8]          1189        0.7435           39    0.0328 1.6212
(0.8, 1.0]         16292        0.8786          212    0.0130 0.6431 

MODEL 2   (random hit rate 2.02%)
               campaigns  success_rate  near_misses  hit_rate   lift
bracket                                                             
(-0.001, 0.2]       1714        0.0583           15    0.0088 0.4325
(0.2, 0.4]          1977        0.2635           52    0.0263 1.3000
(0.4, 0.6]          2542        0.4831          107    0.0421 2.0804
(0.6, 0.8]          3095        0.6691          123    0.0397 1.9642
(0.8, 1.0]         17411        0

### EXPLORATION — Calibration
Model 2 predicts 78.5% of 2024 campaigns will succeed; 72.2% did. **6.4 points optimistic** — slightly
worse than Model 1's 5.7, and for the same reason: both learned on 2019–23, when success averaged 73.5%,
and 2024 was a harder year.

Fitting more parameters did not fix this, which is the point. Drift between training years and the scored
year is not something a model can learn its way out of when it never sees the later year. It does not touch
AUC, but Step 5 reads these scores as real probabilities, so the brackets are sitting about 6 points high.

In [13]:
print(f"{'':<12}{'mean predicted':>16}{'actual':>10}{'optimism':>11}")
print("-" * 49)
for label, s in [("Model 1", score_m1), ("Model 2", score_m2)]:
    print(f"{label:<12}{s.mean():>16.1%}{y_val.mean():>10.1%}{s.mean()-y_val.mean():>+11.1%}")
print(f"\ntraining years averaged {y_train.mean():.1%} success; 2024 came in at {y_val.mean():.1%}")

              mean predicted    actual   optimism
-------------------------------------------------
Model 1                77.9%     72.2%      +5.7%
Model 2                78.5%     72.2%      +6.4%

training years averaged 73.5% success; 2024 came in at 72.2%


### EXPLORATION — Does `blurb_len` earn its column?
Step 2 kept blurb length while noting the raw success rate was flat across it (71–76%), at a cost of one
column. Model 2 settles it: the weight is the smallest of the five numeric inputs (+0.6% odds per 10
characters), and dropping the column entirely changes validation AUC by 0.0001.

**Keep it** — it costs one column out of 191 and Model 4 will want blurb-derived features nearby for
comparison — but it should not appear in any "what drives success" slide. Length of blurb does not matter;
Model 4 will test whether the *content* does.

In [14]:
for drop in [[], ["blurb_len"]]:
    cols = [c for c in X_train.columns if c not in drop]
    sc = StandardScaler().fit(X_train[cols])
    m = LogisticRegression(max_iter=3000, C=C_BEST).fit(sc.transform(X_train[cols]), y_train)
    a = roc_auc_score(y_val, m.predict_proba(sc.transform(X_val[cols]))[:, 1])
    print(f"{'all 191 columns' if not drop else 'without blurb_len (190)':<26} validation AUC = {a:.4f}")

all 191 columns            validation AUC = 0.8526


without blurb_len (190)    validation AUC = 0.8525


### PIPELINE — Save Model 2's outputs
- `model2_val.parquet` — one row per 2024 campaign, so Models 3 and 4 compare on identical rows
- `model2_coefficients.csv` — all 191 weights, the readable artifact and the source for any
  "what drives success" slide

In [15]:
out = pd.DataFrame({"id": val["id"].to_numpy(), "score_model2": score_m2,
                    "success": y_val, "near_miss": nm_val})
out.to_parquet(f"{BASE}/model2_val.parquet", index=False)

co_out = co.copy()
co_out.insert(1, "train_mean", scaler.mean_)
co_out.insert(2, "train_sd", scaler.scale_)
co_out.sort_values("weight_std", ascending=False).to_csv(f"{BASE}/model2_coefficients.csv", index=False)

print(f"saved model2_val.parquet          {out.shape[0]:,} rows")
print(f"saved model2_coefficients.csv     {len(co_out)} weights + intercept {b0:+.4f}")
print()
print(out.head().to_string(index=False))

saved model2_val.parquet          26,739 rows
saved model2_coefficients.csv     191 weights + intercept +4.8354

        id  score_model2  success  near_miss
1702314537      0.207530        1          0
 853086300      0.995642        1          0
 129968201      0.991437        1          0
1171697795      0.986665        1          0
 764915614      0.989370        1          0


### Results — Model 2 summary

| Metric (validation 2024) | Model 1 | **Model 2** | Verdict |
|---|---|---|---|
| AUC | 0.807 | **0.853** | ✅ +0.046 |
| AUC inside "Games / Tabletop Games" | 0.500 | **0.776** | ✅ the source of the gain |
| Distinct scores | 60 | 26,739 | ✅ capacity-sized lists possible |
| Precision of the 2,000-campaign list | 81.6% | **92.5%** | ✅ |
| Near-miss hit rate, riskiest-first | 1.85% | 0.95% | ❌ **worse**, and worse than random |
| Near-miss hit rate, best bracket | 3.76% (931 campaigns) | **4.21% (2,542)** | ✅ 2.08× random |
| Calibration | +5.7 pts | +6.4 pts | ❌ marginally worse |

**Model 2 beats the baseline and becomes the model to beat: AUC 0.853.**

**What to carry into Model 3:**
1. **The gain came from ranking inside categories** — goal size above all. Boosting should extend this by
   finding interactions the linear model cannot, such as a large goal mattering more in Technology than in
   Comics.
2. **Report both metrics, always.** Model 2 improved AUC and made the naive coaching rule worse. A
   single-number comparison would have hidden that completely.
3. **The 40–60% bracket is the working coaching rule** until Step 5 sets it properly on validation.
4. **Drift is unfixed and may not be fixable.** Both models run ~6 points optimistic on 2024. Worth a line
   in the limitations rather than another modelling attempt.

**Limitations.** Regularization strength barely matters (0.0017 across the sweep), so the "tuned" `C` is
cosmetic; all one-hot levels were kept, so category weights read against the average category rather than a
named reference level; `subcategory_other` carries 18.9% of training in a single weight; the train-validation
AUC gap of 0.069 mixes genuine overfitting with year drift and this notebook does not separate them.